# Building a Reproducible Mental Health Data Ecosystem: The Kilifi County, Kenya FAIR² Model Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an mlcroissant.Metadata object

print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @id
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets found in the Croissant schema.")
else:
    print("Available record sets and their @id:")
    for rs in record_sets:
        print(f"  - {rs['@id']} (name: {rs.get('name', 'N/A')})")
    
    # For the first record set, list its fields
    example_rs = record_sets[0]['@id']
    fields = dataset.fields(record_set=example_rs)
    print(f"\nFields in record set {example_rs}:")
    for field in fields:
        field_id = field['@id'] if '@id' in field else 'N/A'
        print(f"  - {field_id} (name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')})")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
dataframes = {}
all_rs_ids = [rs['@id'] for rs in dataset.record_sets]

for record_set_id in all_rs_ids:
    print(f"Extracting records from record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f" - {len(df)} records loaded. Columns: {df.columns.tolist()}")
        if not df.empty:
            display(df.head())
    except Exception as e:
        print(f"Failed to extract records for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, select a numeric field from the first non-empty record set
import numpy as np

target_df = None
target_rs_id = None
numeric_field_id = None
group_field_id = None

# Try to detect a likely numeric field (e.g., contains 'score', 'age', etc.)
for rs_id, df in dataframes.items():
    if not df.empty:
        possible_numeric = [c for c in df.columns if 'score' in c.lower() or df[c].dtype in [np.float64, np.int64, 'float64', 'int64'] or df[c].map(lambda x: str(x).isdigit()).all()]
        if possible_numeric:
            target_df = df.copy()
            target_rs_id = rs_id
            numeric_field_id = possible_numeric[0]
            # Try to find a group field (e.g., gender, education, village, etc.)
            group_candidates = [c for c in df.columns if c.lower() in ['gender', 'village', 'level_of_education', 'marital_status', 'age']]
            if group_candidates:
                group_field_id = group_candidates[0]
            break

if target_df is not None and numeric_field_id is not None:
    # Make sure numeric field is numeric
    target_df[numeric_field_id] = pd.to_numeric(target_df[numeric_field_id], errors='coerce')

    print(f"Using record set: {target_rs_id}\nNumeric field for analysis: {numeric_field_id}")

    threshold = target_df[numeric_field_id].mean() if not target_df[numeric_field_id].isnull().all() else 0
    # Filter for values above mean (ex: flag high symptom scores)
    filtered_df = target_df[target_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field for these filtered records
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the group_field, if available
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if target_df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(target_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id is not None and group_field_id in target_df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=target_df[group_field_id], y=target_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Visualization not possible due to insufficient numeric data.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We used the `mlcroissant` library to access a FAIR²-structured mental health dataset for Kilifi County, Kenya.
- Record sets and fields were programmatically listed and explored using their Croissant `@id` identifiers.
- Example EDA was performed on a numeric survey field, demonstrating filtering, normalization, and grouping.
- Visualization provided insights into the data distribution and demographic breakdowns (where possible).

For further analysis, consult the field and record set `@id`s via the notebook above and adapt extractions to your goals.